# BÀI TẬP CHƯƠNG 2
**Họ tên:** Vũ Gia Huy &nbsp;|&nbsp; **MSSV:** 23174600110  
**Môn:** Lập trình Data Science &nbsp;|&nbsp; **GV:** Lê Hằng Anh

In [ ]:
import sqlite3
import pandas as pd
from datetime import datetime, timedelta

conn   = sqlite3.connect(':memory:')
cursor = conn.cursor()

cursor.execute('''
CREATE TABLE student (
    student_id INTEGER PRIMARY KEY,
    name       TEXT,
    class      TEXT,
    course_id  INTEGER,
    score      REAL
)''')

cursor.execute('''
CREATE TABLE course (
    id          INTEGER PRIMARY KEY,
    course_name TEXT
)''')

cursor.executemany('INSERT INTO student VALUES (?,?,?,?,?)', [
    (1,  'Nguyen Minh Hoang', 'May Tinh', 12,   6.7),
    (2,  'Tran Thi Lan',      'Kinh Te',  34,   9.2),
    (3,  'Pham Van Nam',      'Toan Tin', None, 7.9),
    (4,  'Le Thanh Huyen',    'Toan Tin', 20,   7.2),
    (5,  'Vu Quoc Anh',       'May Tinh', 24,   8.0),
    (6,  'Dang Thuy Linh',    'May Tinh', 24,   5.5),
    (7,  'Bui Tien Dung',     'Kinh Te',  34,   9.2),
    (8,  'Ho Ngoc Mai',       'Toan Tin', 20,   8.8),
    (9,  'Duong Huu Phuc',    'Kinh Te',  None, 7.2),
    (10, 'Cao Thi Hanh',      'May Tinh', None, 7.0),
])

cursor.executemany('INSERT INTO course VALUES (?,?)', [
    (12, 'Giai tich'),
    (34, 'Thong ke'),
    (26, 'Tin hoc'),
])
conn.commit()

def sql(query):
    return pd.read_sql_query(query, conn)

## Câu 1 – Kết nối hai bảng

In [ ]:
# Tích Descartes
sql('''
    SELECT s.student_id, s.name, s.class, s.course_id, s.score,
           c.id AS c_id, c.course_name
    FROM student s
    CROSS JOIN course c
''')

In [ ]:
# INNER JOIN
sql('''
    SELECT s.student_id, s.name, s.class, s.course_id, s.score,
           c.course_name
    FROM student s
    INNER JOIN course c ON s.course_id = c.id
''')

In [ ]:
# LEFT JOIN
sql('''
    SELECT s.student_id, s.name, s.class, s.course_id, s.score,
           c.course_name
    FROM student s
    LEFT JOIN course c ON s.course_id = c.id
''')

In [ ]:
# RIGHT JOIN (SQLite không hỗ trợ trực tiếp -> đảo bảng dùng LEFT JOIN)
sql('''
    SELECT s.student_id, s.name, s.class, s.course_id, s.score,
           c.id AS c_id, c.course_name
    FROM course c
    LEFT JOIN student s ON s.course_id = c.id
''')

In [ ]:
# FULL OUTER JOIN (mo phong bang UNION)
sql('''
    SELECT s.student_id, s.name, s.class, s.course_id, s.score,
           c.id AS c_id, c.course_name
    FROM student s
    LEFT JOIN course c ON s.course_id = c.id

    UNION

    SELECT s.student_id, s.name, s.class, s.course_id, s.score,
           c.id AS c_id, c.course_name
    FROM course c
    LEFT JOIN student s ON s.course_id = c.id
''')

## Câu 2 – Cập nhật course_id và phân tích

In [ ]:
# Dien course_id NULL bang course_id pho bien nhat (hop le) trong cung lop
cursor.execute('''
UPDATE student
SET course_id = (
    SELECT s2.course_id
    FROM student s2
    INNER JOIN course c ON s2.course_id = c.id
    WHERE s2.class = student.class
    GROUP BY s2.course_id
    ORDER BY COUNT(*) DESC
    LIMIT 1
)
WHERE course_id IS NULL
''')

# Xoa ban ghi co course_id khong ton tai trong bang course
cursor.execute('''
DELETE FROM student
WHERE course_id NOT IN (SELECT id FROM course)
   OR course_id IS NULL
''')
conn.commit()

sql('''
    SELECT s.*, c.course_name
    FROM student s
    JOIN course c ON s.course_id = c.id
    ORDER BY student_id
''')

In [ ]:
# 2a. Tong so sinh vien va diem trung binh theo lop
sql('''
    SELECT
        class                   AS "Lop",
        COUNT(*)                AS "Tong SV",
        ROUND(AVG(score), 2)    AS "Diem TB"
    FROM student
    GROUP BY class
    ORDER BY "Diem TB" DESC
''')

In [ ]:
# 2b. Tong so sinh vien va diem trung binh theo mon hoc
sql('''
    SELECT
        c.course_name           AS "Mon hoc",
        COUNT(*)                AS "Tong SV",
        ROUND(AVG(s.score), 2)  AS "Diem TB"
    FROM student s
    JOIN course c ON s.course_id = c.id
    GROUP BY c.course_name
    ORDER BY "Diem TB" DESC
''')

In [ ]:
# 2c. Phan loai thi dua theo diem TB tung mon
sql('''
    SELECT
        c.course_name                   AS "Mon hoc",
        ROUND(AVG(s.score), 2)          AS "Diem TB",
        CASE
            WHEN AVG(s.score) >= 9.0    THEN 'Xuat sac'
            WHEN AVG(s.score) >= 6.0    THEN 'Tot'
            ELSE                             'Kem'
        END                             AS "Xep loai"
    FROM student s
    JOIN course c ON s.course_id = c.id
    GROUP BY c.course_name
    ORDER BY "Diem TB" DESC
''')

## Câu 3 – Xếp hạng sinh viên

In [ ]:
# 3a. Xep hang theo diem so toan truong
df3a = sql('''
    SELECT name, class, score,
           RANK() OVER (ORDER BY score DESC) AS rank_overall
    FROM student
    ORDER BY rank_overall
''')
display(df3a)
display(df3a.nsmallest(3, 'rank_overall').reset_index(drop=True))
display(df3a.nlargest(3,  'rank_overall').reset_index(drop=True))

In [ ]:
# 3b. Xep hang theo diem so trong tung lop
df3b = sql('''
    SELECT name, class, score,
           RANK() OVER (PARTITION BY class ORDER BY score DESC) AS rank_in_class
    FROM student
    ORDER BY class, rank_in_class
''')
display(df3b)
display(df3b.groupby('class').apply(lambda g: g.nsmallest(1, 'rank_in_class')).reset_index(drop=True))
display(df3b.groupby('class').apply(lambda g: g.nlargest(1,  'rank_in_class')).reset_index(drop=True))

In [ ]:
# 3c. Xep hang theo diem so trong tung mon hoc
df3c = sql('''
    SELECT s.name, c.course_name, s.score,
           RANK() OVER (PARTITION BY s.course_id ORDER BY s.score DESC) AS rank_in_course
    FROM student s
    JOIN course c ON s.course_id = c.id
    ORDER BY c.course_name, rank_in_course
''')
display(df3c)
display(df3c.groupby('course_name').apply(lambda g: g.nsmallest(1, 'rank_in_course')).reset_index(drop=True))
display(df3c.groupby('course_name').apply(lambda g: g.nlargest(1,  'rank_in_course')).reset_index(drop=True))

## Câu 4 – Bổ sung trường `graduation_date`

In [ ]:
cursor.execute('ALTER TABLE student ADD COLUMN graduation_date DATETIME')

now   = datetime.now()
ranks = sql('SELECT student_id, RANK() OVER (ORDER BY score DESC) AS rnk FROM student')

for _, row in ranks.iterrows():
    grad = (now + timedelta(days=int(row['rnk']))).strftime('%Y-%m-%d %H:%M:%S')
    cursor.execute(
        'UPDATE student SET graduation_date = ? WHERE student_id = ?',
        (grad, int(row['student_id']))
    )
conn.commit()

sql('''
    SELECT s.student_id, s.name, s.class, c.course_name, s.score,
           RANK() OVER (ORDER BY s.score DESC) AS rank_overall,
           s.graduation_date
    FROM student s
    JOIN course c ON s.course_id = c.id
    ORDER BY rank_overall
''')